# 02 — Policy Abstraction

The `Policy` ABC is the core contract. Every VLA provider implements it,
so robot code never changes when you swap models.

## The ABC

```python
class Policy(ABC):
    @abstractmethod
    async def get_actions(self, observation_dict, instruction, **kwargs) -> list[dict[str, Any]]: ...

    def get_actions_sync(self, observation_dict, instruction, **kwargs) -> list[dict[str, Any]]: ...

    @abstractmethod
    def set_robot_state_keys(self, robot_state_keys: list[str]) -> None: ...

    @property
    @abstractmethod
    def provider_name(self) -> str: ...
```

Design decisions:
- **Async-first** — `get_actions()` is async for real robot event loops
- **Sync wrapper** — `get_actions_sync()` auto-detects event loop, uses ThreadPoolExecutor fallback
- **Observation = dict** — `{camera_name: ndarray, joint_name: float}`
- **Actions = list[dict]** — each dict is one timestep, keys are actuator names

In [ ]:
from strands_robots.policies import Policy, MockPolicy, create_policy

# MockPolicy generates deterministic sinusoidal trajectories
mock = MockPolicy()
print(f"provider: {mock.provider_name}")

mock.set_robot_state_keys(["shoulder", "elbow", "wrist", "gripper"])
print(f"state_keys: {mock.robot_state_keys}")

In [ ]:
# get_actions_sync — safe from sync code, event loops, or notebooks
obs = {"shoulder": 0.0, "elbow": 0.5, "wrist": -0.3, "gripper": 0.0}
actions = mock.get_actions_sync(obs, "pick up the cube")

print(f"chunk size: {len(actions)}")  # always 8
print(f"keys: {list(actions[0].keys())}")
print(f"first step: { {k: round(v, 4) for k, v in actions[0].items()} }")

# Values are smooth sinusoids — freq and phase vary per joint index
print(f"shoulder over 8 steps: {[round(a['shoulder'], 3) for a in actions]}")

In [ ]:
# Internal step counter increments — actions change between calls
actions2 = mock.get_actions_sync(obs, "pick up the cube")
print(f"same first step? {actions[0] == actions2[0]}")  # False — step counter advanced

## `create_policy()` Factory

Resolves providers from `registry/policies.json`. Accepts:
- Provider names: `"mock"`, `"groot"`, `"lerobot_local"`
- Smart strings: `"lerobot/act_aloha_sim"` → auto-detects provider
- URL patterns: `"zmq://localhost:5555"` → routes to GR00T

In [ ]:
from strands_robots.policies import create_policy, list_providers

print(f"providers: {list_providers()}")

# Create by name
m = create_policy("mock")
print(f"created: {m.provider_name}")

## Writing a Custom Policy

In [ ]:
import numpy as np
from strands_robots.policies import Policy, register_policy

class RandomPolicy(Policy):
    """Uniform random actions — simplest possible policy."""

    def __init__(self, scale: float = 0.1, chunk_size: int = 8, **kwargs):
        self.scale = scale
        self.chunk_size = chunk_size
        self._keys: list[str] = []

    @property
    def provider_name(self) -> str:
        return "random"

    def set_robot_state_keys(self, keys: list[str]) -> None:
        self._keys = keys

    async def get_actions(self, observation_dict, instruction, **kwargs):
        return [
            {k: float(np.random.uniform(-self.scale, self.scale)) for k in self._keys}
            for _ in range(self.chunk_size)
        ]

# Register so create_policy("random") works
register_policy("random", lambda: RandomPolicy, aliases=["rnd"])

rnd = create_policy("rnd", scale=0.05)
rnd.set_robot_state_keys(["j1", "j2", "j3"])
print(create_policy("random").provider_name)
print(rnd.get_actions_sync({}, "test")[0])